10 wav files; poor results. Manual individual selection of time indices

In [1]:
from lib import epl 
from lib import OSN
from lib import readout
from lib import plots
import time
import numpy as np
import pickle 
import brian2 as b2
import brian2hears as b2h
import matplotlib.pyplot as plt
#import csv
import os.path
from os import path
from numpy.matlib import repmat
%matplotlib inline
from preprocess import preprocessing as pre

In [2]:
testSounds = ["00f0204f_nohash_2", "00970ce1_nohash_0", "004ae714_nohash_0", "004ae714_nohash_1", "0132a06d_nohash_0", "0132a06d_nohash_2", "0132a06d_nohash_4", "0135f3f2_nohash_1", "0137b3f4_nohash_0", "0137b3f4_nohash_1"]              
trainingSounds = ["00f0204f_nohash_2"]
scale = 15 

trainingIndex = [4000,6000]
testingIndex = [4000, 4500, 8000, 7500, 6000, 5500, 8000, 5500, 4500, 7500]
range_f = 100
step_f = 10
cfN = 20 #channel frequency number
colors = ['y','k','b','y','k','b','y','k','b','y']

trainingAudio = pre.createTraining(trainingSounds, trainingIndex, cfN, range_f, step_f, scale)
testAudio = pre.createTesting(testSounds, cfN, testingIndex, colors, range_f, step_f, scale)

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/micha/A-Neuromorphic-App-to-Keyword-Recog-main/A-Neuromorphic-App-to-Keyword-Recog-main/yes_sounds/00f0204f_nohash_2.wav'

In [ ]:
trainingAudio.shape

In [ ]:
#Plot histogram of frequency vs amplitude for all training+testSounds

# Combine and average
allSounds = np.concatenate((trainingAudio, testAudio), axis=0)
#data = np.mean(allSounds, axis=0)
ind = 0
data = testAudio[ind]

# Generate frequency axis (assuming evenly spaced bins from 0 to 200 Hz)
freqs = np.linspace(0, 200, allSounds.shape[1])

plt.plot(freqs, allSounds[[0,1]].T, label='training sounds')
plt.plot(freqs, data.T, label='testing sound')

plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude (Averaged Across Sounds)")
plt.title("Frequency vs Amplitude Histogram")

interStep=1#int(allSounds.shape[1]/50)
print("interpolation step size: "+str(interStep))
interFreqs = np.linspace(0, 200, interStep)

#plt.plot(interFreqs-0, np.interp(interFreqs, testAudio[ind], data), label='Interpolation of ind test sound')
plt.plot(interFreqs-0, np.interp(interFreqs, freqs, data), label='Interpolation of ind test sound')

plt.legend()
plt.show()

sound = testSounds[ind]
testingSpect = pre.spectArray(sound, cfN=cfN)
plt.axvline(x=testingIndex[testSounds.index(sound)], c=colors[testSounds.index(sound)])
pre.spectShow(testingSpect)


In [ ]:
trainingAudioSpec = [pre.spectArray(sound) for sound in trainingSounds]
testAudioSpec = [pre.spectArray(sound) for sound in testSounds]

target_length = 16000  # or max(x.shape[1] for x in testAudioSpec)

for i in range(0,len(testAudioSpec)):
    x = testAudioSpec[i]
    print(x.shape[1])
    if x.shape[1] < target_length:
        
        # Pad on the right
        pad_width = target_length - x.shape[1]
        x = np.pad(x, ((0, 0), (0, pad_width)), mode='constant')
        
    elif x.shape[1] > target_length:
        # Trim on the right
        x = x[:, :target_length]
    testAudioSpec[i] = x


In [ ]:
# Reference waveform (e.g., first training example)
ref = trainingAudioSpec[0]

# Aligned list (start with reference itself)
aligned = [ref]

# Loop over other waveforms
for i in range(0, len(testAudioSpec)):
    x = testAudioSpec[i]
    corr = correlate(x, ref, mode='full')
    lag = np.argmax(corr) - (len(x) - 1)
    
    # Shift signal using np.roll
    shifted = np.roll(x, -lag)
    #print(shifted.shape)
    aligned.append(shifted)

# Convert to NumPy array
alignedAudio = np.array(aligned)


In [ ]:
from lib import epl
nGammaPerTraining = 5
nGammaPerSound = 5  #for this dataset nGammaPerSound = 5 is not enough for differentiation as shown in SImatrix
nAudios = len(trainingAudio); 
lenTraining=nAudios
lenTest=len(testAudio)
nTestPerAudio = len(testAudio)/nAudios;  
print("Number of odors to train = " + str(lenTraining)); 
print("Number of odors to test = " + str(lenTest)); 

#Network initialization
nMCs = len(trainingAudio[0]); 
GCsPerNeurogenesis = 5; 
nGCs = nMCs*GCsPerNeurogenesis*lenTraining;     #every MC has 5 GCs per sound  
epl = epl.EPL(nMCs, nGCs, GCsPerNeurogenesis); 

#Sniff
def sniff(sound, learn_flag=0, nGammaPerSound=5, gPeriod=40):
    sensorInput = OSN.OSN_encoding(sound); 
    for j in range(0, nGammaPerSound):
        for k in range(0, gPeriod): 
            epl.update(sensorInput, learn_flag=learn_flag);
            pass; 
    epl.reset(); 

#Training
t1 = time.time(); 
for i in range(0, nAudios):
    print("Training sound " + str(i+1)) 
    sniff(trainingAudio[i], learn_flag=1,nGammaPerSound = nGammaPerTraining); 
    # changed nGammaPerSound from 5 to 10
    epl.GClayer.invokeNeurogenesis();
#added this for loop to change structure of epl.gammaCode
for i in range(0, lenTraining):
    sniff(trainingAudio[i], learn_flag=0, nGammaPerSound = nGammaPerSound); #try nGammaPerTraining
    
#Testing
for i in range(0, lenTest):
    sniff(testAudio[i], learn_flag=0, nGammaPerSound = nGammaPerSound); 
    if((i+1)%10==0):
        print(str(i+1) + " sounds tested"); 
t2 = time.time(); 
print("Simulation Duration = " + str(t2-t1) + "s")

In [ ]:
from my_lib import my_readout
sMatrix = my_readout.readout(epl.gammaCode, nAudios, nTestPerAudio)  

In [ ]:
plotTitles= ['Sound 1', 'Sound 2', 'Sound 3', 'Sound 4', 'Sound 5']
plotFigure4b_audio(sMatrix)
plotFigure4d_audio(epl.gammaCode, sMatrix)

In [ ]:
def plotFigure4d_audio(gammaCode, sMatrix):
    fig, axs = plt.subplots(figsize=(18, 9))
    fig.suptitle("Figure 4d - Spectogram-sliced Audio", fontsize=16)

    # Use digits you have
    soundNum = ['Sound 1', 'Sound 2']
    nOdors = len(soundNum)
    plotTitles = [d for d in soundNum]

    # Adjust plotting order if needed
    nMCs = len(gammaCode[0])
    nGamma = 5
    nTestPerOdor = int(len(sMatrix) / (nOdors * nGamma))

    plt.subplots_adjust(hspace=0.45, wspace=0.25)
    fsize = 12
    fsize2 = 16

    for i in range(nOdors):
        # Raster plot
        plt.subplot(3, 4, 2 * i + 1)
        odorID = i

        sniffID = nOdors * 2 + nTestPerOdor * odorID
        print("sniffID: "+str(sniffID))
        for j in range(nGamma):
            gammaID = int(nGamma * sniffID + j)
            print("gammaID: " +str(gammaID))
            for k in range(nMCs):
                if gammaCode[gammaID][k] != 0:
                    spikeTime = 20 - gammaCode[gammaID][k] + 40 * j
                    #
                    plt.scatter(spikeTime, k, color='k', s=2)
        plt.title(plotTitles[i], fontsize=fsize2)
        plt.yticks([0, 20, 40, 60], fontsize=fsize)
        if i >= (nOdors - 2):
            plt.xticks([0, 40, 80, 120, 160, 200], fontsize=fsize)
            plt.xlabel('Timesteps', fontsize=fsize2)
            plt.ylabel('MC Index', fontsize=fsize2)
        else:
            plt.xticks([])

        # Similarity plot
        plt.subplot(3, 4, 2 * i + 2)
        bar_y = []
        gammaID = odorID * nTestPerOdor * nGamma
        for j in range(5):
            #print(int(gammaID + j))
            #print(sMatrix[int(gammaID + j)][odorID])
            bar_y.append(sMatrix[int(gammaID + j)][odorID])
            
            
        plt.bar([40, 80, 120, 160, 200], bar_y, color='black', width=10.0)
        plt.yticks([0.2, 0.4, 0.6, 0.8, 1.0], fontsize=fsize)
        plt.ylim([0.0, 1.02])
        plt.xlim([0, 240])
        if i >= (nOdors - 2):
            plt.xticks([40, 80, 120, 160, 200], [1, 2, 3, 4, 5], fontsize=fsize)
            plt.xlabel('Gamma Cycles', fontsize=fsize2)
            plt.ylabel('Similarity', fontsize=fsize2)
        else:
            plt.xticks([])
    plt.show()
def plotFigure4b_audio(sMatrix): 
    bar1_x, bar2_x, bar3_x, bar4_x, bar5_x = [], [], [], [], [];      
    bar1_y, bar2_y, bar3_y, bar4_y, bar5_y = [], [], [], [], [];      
    nGamma = 5; 
    testOdorID = 0;              #sniff ID of test odor 
    for i in range(0, nGamma):
        bar1_y.append(sMatrix[testOdorID+i][0]);
        #bar2_y.append(sMatrix[testOdorID+i][4]);
        #bar3_y.append(sMatrix[testOdorID+i][5]);
        bar4_y.append(sMatrix[testOdorID+i][1]);
        #bar5_y.append(sMatrix[testOdorID+i][2]);
    w = 0.15        #width of bars
    fsize = 14;
    xticks = []
    xtick_labels = []
    for i in range(0, nGamma):    
        bar1_x.append(i-2*w);
        bar2_x.append(i-w);
        bar3_x.append(i);
        bar4_x.append(i+w); 
        bar5_x.append(i+2*w);     
        xticks.append(i)
        xtick_labels.append(str(i+1))
    plt.figure(2, figsize=(10, 5))
    fig = plt.subplot(111)
    opacity = 0.5; 
    bar1 = plt.bar(bar1_x, bar1_y, width = w, color = 'blue', alpha = opacity, align='center')
    #bar2 = plt.bar(bar2_x, bar2_y, width = w, color = 'violet', alpha = opacity, align='center')
    #bar3 = plt.bar(bar3_x, bar3_y, width = w, color = 'red', alpha = opacity, align='center')
    bar4 = plt.bar(bar4_x, bar4_y, width = w, color = 'orange', alpha=opacity, align='center')
    #bar5 = plt.bar(bar5_x, bar5_y, width = w, color = 'green', alpha=opacity, align='center')
    plt.xlim(-1, len(bar1_x)+0.2)
    plt.ylim(0, 1.02)
    plt.xticks(xticks, xtick_labels, fontsize=fsize)
    plt.yticks(fontsize=fsize); 
    soundNum = ['Sound 1', 'Sound 2']
    fig.legend( (bar1,  bar4), soundNum, loc = 'upper left', fontsize=fsize)
    plt.ylabel('Similarity', fontsize=fsize); 
    plt.xlabel('Gamma Cycles', fontsize=fsize); 
    plt.title("Figure 4b")
    plt.show()

## epl gammaCode parsing

if I keep nGammaPerTraining in the second training loop, I need to change it in epl gammaCode parsing as well

In [ ]:
#Remember the first training loop uses nGammaPerSound = 10

gammaCode_size = lenTraining*(nGammaPerTraining + nGammaPerTraining) + lenTest * nGammaPerSound
(gammaCode_size,len(epl.gammaCode))

In [ ]:
trained = np.array(epl.gammaCode[lenTraining*nGammaPerTraining:lenTraining*(nGammaPerTraining+nGammaPerSound)]).reshape(nGammaPerSound, lenTraining,-1)#/nGammaPerSound, -1)
#put lenTest/nGammaPerSound into another variable
tested = np.array(epl.gammaCode[lenTraining*(nGammaPerTraining+nGammaPerSound):]).reshape(nGammaPerSound, lenTest, -1)

In [ ]:

true_negative = 0
false_positive = 0


SImatrix = np.zeros((lenTraining, lenTest), dtype =float)  
for i in range(0, len(trained[-1])):
    for j in range(0, len(tested[-1])):
        SImatrix[i][j] = (pre.computeSimilarity(trained[-1][i], tested[-1][j]))
decisionMatrix = (SImatrix> 0.75)

In [ ]:
#designing truth Table
truthTable = np.tile([True, False], (lenTest,1)).T

In [ ]:
truthTable

In [ ]:
decisionMatrix

In [ ]:
SImatrix

In [ ]:
for l in (decisionMatrix==truthTable):
    true_negative += sum((decisionMatrix==truthTable).reshape((1,20)))

In [ ]:
sum((decisionMatrix==truthTable).reshape((1,20)))

In [ ]:
decisionMatrix==truthTable

In [ ]:
pre.results(decisionMatrix, truthTable)

In [ ]:
plt.plot([trainingIndex[0]]*lenTest , SImatrix.T, ".") 
plt.ylabel('Similarity')
plt.xlabel('Time (index)')


In [ ]:
plt.plot(range(1,11) , SImatrix.T, ".") 
